# MulViT-TWDNN Fine-tuning
Shared ViT + Token-wise DNN Fusion

## Model

In [ ]:
class ViTEncoder(nn.Module):
    """Compatible with MAE/CLS pretrained weights — ViT backbone (embed_dim=96, depth=6)"""
    def __init__(self, pretrained_path):
        super().__init__()
        self.vit = VisionTransformer(
            img_size=(IMG_H, IMG_W), patch_size=16,
            embed_dim=96, depth=6, num_heads=3,
            mlp_ratio=4., qkv_bias=True, num_classes=0)
        if pretrained_path and os.path.isfile(pretrained_path):
            ckpt  = torch.load(pretrained_path, map_location='cpu', weights_only=False)
            state = ckpt.get('model', ckpt)
            vit_s = {}
            for k, v in state.items():
                if   k.startswith('encoder.'): vit_s[k[8:]] = v
                elif k.startswith('vit.'):     vit_s[k[4:]] = v
                else:                          vit_s[k]     = v
            miss, unex = self.vit.load_state_dict(vit_s, strict=False)
        else:
            pass
        self.vit.pos_embed = nn.Parameter(self.vit.pos_embed.data.clone())

    @property
    def learnable_token(self): return self.vit.cls_token
    @property
    def pos_embed(self):       return self.vit.pos_embed
    @property
    def patch_embed(self):     return self.vit.patch_embed
    @property
    def blocks(self):          return self.vit.blocks
    @property
    def norm(self):            return self.vit.norm

    def forward_features(self, x):
        """Full token sequence [B, 301, 96]  (15x20 patches + CLS)"""
        x = self.vit.patch_embed(x)
        x = self.vit._pos_embed(x)
        if hasattr(self.vit, 'patch_drop'): x = self.vit.patch_drop(x)
        if hasattr(self.vit, 'norm_pre'):   x = self.vit.norm_pre(x)
        x = self.vit.blocks(x)
        x = self.vit.norm(x)
        return x  # [B, 301, 96]

    def forward(self, x): return self.vit(x)  # [B, 96]

class DNNResidualBlock(nn.Module):
    """
    Token-wise DNN Residual Block
    [B, L, 96] -> LN -> Linear(96->192) -> GELU -> Linear(192->192) -> GELU -> Linear(192->96) + residual
    Shared weights applied token-wise
    """
    def __init__(self, dim=96, hidden_dim=192, dropout=DROPOUT):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.mlp  = nn.Sequential(
            nn.Linear(dim,        hidden_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, dim),                   nn.Dropout(dropout))

    def forward(self, x):   # x: [B, L, dim]
        return x + self.mlp(self.norm(x))

class DNNFusion(nn.Module):
    """
    [tokens1 || tokens2] [B, 602, 96]
    -> seg_embed -> DNNResidualBlock x NUM_LAYERS
    -> CLS1=x[:,0], CLS2=x[:,301] -> concat [B, 192]
    """
    def __init__(self, dim=96, hidden_dim=192, dropout=DROPOUT, num_layers=NUM_LAYERS):
        super().__init__()
        self.seg_embed = nn.Parameter(torch.zeros(1, 2, dim))
        nn.init.trunc_normal_(self.seg_embed, std=0.02)
        self.blocks   = nn.Sequential(
            *[DNNResidualBlock(dim, hidden_dim, dropout) for _ in range(num_layers)])
        self.out_norm = nn.LayerNorm(dim * 2)

    def forward(self, tokens1, tokens2):
        t1 = tokens1 + self.seg_embed[:, 0:1]   # [B, 301, 96]
        t2 = tokens2 + self.seg_embed[:, 1:2]
        x  = torch.cat([t1, t2], dim=1)          # [B, 602, 96]
        x  = self.blocks(x)
        cls1 = x[:, 0]
        cls2 = x[:, 301]
        return self.out_norm(torch.cat([cls1, cls2], dim=-1))  # [B, 192]

class MulViTTWDNN(nn.Module):
    """cam1 + cam2 -> shared ViT -> DNNFusion [B,192] -> Linear head -> RSSI"""
    def __init__(self, pretrained_path, dropout=DROPOUT):
        super().__init__()
        self.vit    = ViTEncoder(pretrained_path)   # single encoder shared across cameras
        self.fusion = DNNFusion(dim=96, hidden_dim=192, dropout=dropout, num_layers=NUM_LAYERS)
        self.head   = nn.Sequential(
            nn.LayerNorm(192), nn.Dropout(dropout),
            nn.Linear(192, 128), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(128, 1))

    def forward(self, img1, img2):
        t1 = self.vit.forward_features(img1)
        t2 = self.vit.forward_features(img2)
        return self.head(self.fusion(t1, t2))

    def param_groups(self, lr, backbone_scale):
        bb = {f'vit.vit.{sub}' for sub in ['patch_embed', 'blocks', 'norm']}
        bp, op = [], []
        for name, param in self.named_parameters():
            if '.'.join(name.split('.')[:3]) in bb: bp.append(param)
            else: op.append(param)
        return [{'params':bp,'lr':lr*backbone_scale},{'params':op,'lr':lr}]
